In [ ]:
!pip install spacy bert-score pandas
!python -m spacy download en_core_web_sm

In [ ]:
import json
import spacy
import re

In [ ]:
# nettoyag des réponse du LLM
def nettoyer_texte_llm(texte):
    """
    Nettoie les réponses des LLMs en supprimant les introductions
    et les listes de modifications à la fin.
    """
    if not isinstance(texte, str) or not texte:
        return ""

    texte = texte.strip()

    # Supprimer l'introduction du LLM (préfixe)
    pattern_intro = (
        r"^(?:here(?:'s| is)|sure|certainly|below|this is|as requested|"
        r"of course|absolutely|happy to).*?(?::|\.)\s*\n+"
    )
    texte = re.sub(pattern_intro, "", texte, flags=re.IGNORECASE)

    # Supprimer la conclusion / liste de changements (suffixe)
    pattern_outro = (
        r"(\n\s*(?:i made the following changes|changes made|"
        r"here are the changes|notes?):).*$"
    )
    texte = re.sub(pattern_outro, "", texte, flags=re.IGNORECASE | re.DOTALL)

    return texte.strip()


# CLASSE D'EXTRACTION
class ExtracteurFactuel:

    # Constantes de classe (compilées une seule fois)
    _NER_BLACKLIST  = {'linear', 'minima', 'maxima', 'kernel', 'al.', 'al'}
    _SIGLES_EXCLUS  = {
        'MoCA', 'CaMeLS', 'LLaMa', 'LLaMA',
        'U1', 'U2', 'U3', 'E1', 'E2', 'E3',
        'C1', 'C2', 'C3', 'C4', 'Age2',
    }
    _PATTERN_VAR_SEULE = re.compile(r'^[A-Z]\d{1,2}$')
    _GRECQUES = 'αβγδεζηθικλμνξπρστυφχψωΑΒΓΔΕΖΗΘΙΚΛΜΝΞΠΡΣΤΥΦΧΨΩ'

    def __init__(self):
        print("Chargement du modèle spaCy (en_core_web_sm)...")
        self.nlp = spacy.load("en_core_web_sm")
        print("Modèle chargé !")

    def extraire(self, texte):
        if not texte:
            return {k: set() for k in [
                "Noms_Propres", "Mesures", "References",
                "Molecules_Chimiques", "Equations",
                "Formules_LaTeX", "Formules_Inline"
            ]}

        # noms propres — spaCy + post-filtre

        doc = self.nlp(texte)
        noms_propres = set()
        for ent in doc.ents:
            if ent.label_ not in ["PERSON", "ORG", "GPE", "LOC", "PRODUCT", "WORK_OF_ART"]:
                continue
            t = ent.text.strip()
            if len(t) > 60 or len(t.split()) > 6:
                continue
            if re.search(r'[·\?\!\n\u2019\u201c\u201d]', t):
                continue
            if t and t[0].islower():
                continue
            if t.lower() in self._NER_BLACKLIST:
                continue
            noms_propres.add(t)

        # mesures — unité obligatoire (deux patterns utilisés)

        # unités alphanumériques
        _pat_alpha = (
            r'\b(\d+(?:[.,]\d+)?)\s*'
            r'(mg|kg|µg|g|ml|kHz|MHz|mHz|Hz|kV|mV|ms|µs|mol|mmol|µmol|'
            r'nM|µM|mM|cm|mm|µm|nm|a\.u\.|kDa|Da|rpm|mbar|bar|atm|'
            r'kcal|kJ|eV|keV|MeV|ppm|ppb|ps|ns)\b'
        )
        # unités symboliques
        _pat_sym = (
            r'\b(\d+(?:[.,]\d+)?)\s*'
            r'(%|°C|°F|K|mhz|khz|hz)(?=[\s,\.;\)\n]|$)'
        )
        mesures = set()
        for num, unit in re.findall(_pat_alpha, texte, re.IGNORECASE):
            mesures.add(f"{num} {unit.lower()}")
        for num, unit in re.findall(_pat_sym, texte, re.IGNORECASE):
            mesures.add(f"{num} {unit.lower()}")

        # --------------------------------------------------
        # références bibliographiques
        # --------------------------------------------------
        # (Smith, 2020) | (Smith & Jones, 2020) | (Wu et al., 2018, 2021)
        _pat_ref_paren = (
            r'\([A-Z][a-zA-Z\-]+'
            r'(?:\s+(?:&|and)\s+[A-Z][a-zA-Z\-]+)?'
            r'(?:\s+et\s+al\.?)?'
            r',\s*\d{4}(?:\s*[,;]\s*\d{4})*\)'
        )
        # [1] | [1, 2, 3] | [24,37]
        _pat_ref_croch = r'\[\s*\d+(?:\s*[,;]\s*\d+)*\s*\]'
        references = set(
            re.findall(_pat_ref_paren, texte) +
            re.findall(_pat_ref_croch, texte)
        )

        # --------------------------------------------------
        # molécules chimiques et autres procédés chimiques
        # --------------------------------------------------
        # règle 1 : Majuscule + au moins un chiffre → H2O, CO2, K562, A549
        _mol_chiffres = r'\b[A-Z][a-zA-Z]*\d+[a-zA-Z0-9]*\b'
        # Règle 2 : 2 éléments collés sans chiffres → NaCl, HCl, FeMo
        _mol_lettres  = (
            r'\b[A-Z][a-z][A-Z][a-z]?\b'
            r'|\b(?:NaCl|HCl|FeMo|MgCl2|TiO2|NaOH|NaHCO3|KCl|CaCl2)\b'
        )
        # règle 3 : formules avec parenthèses → Ca(OH)2, Fe2(SO4)3
        _mol_paren = (
            r'\b[A-Z][a-z]?\d*'
            r'(?:\([A-Z][a-z]?\d*(?:[A-Z][a-z]?\d*)*\)\d+)+'
            r'(?:[A-Z][a-z]?\d*)*\b'
        )
        molecules = set()
        for m in (re.findall(_mol_chiffres, texte) +
                  re.findall(_mol_lettres,  texte) +
                  re.findall(_mol_paren,    texte)):
            if self._PATTERN_VAR_SEULE.match(m) and m not in {'H2', 'O2', 'N2', 'CO2', 'NO2'}:
                continue
            if m in self._SIGLES_EXCLUS:
                continue
            molecules.add(m)

        # --------------------------------------------------
        # équations compactes (sans espaces, courtes)
        # --------------------------------------------------
        _pat_eq = (
            r'\b([a-zA-Z_]\w{0,15})'
            r'\s*=\s*'
            r'([-+]?[a-zA-Z0-9_\.\^\*\/\(\)]{1,25})'
            r'(?=[\s,;:\.\n\)\]|]|$)'
        )
        equations = set()
        for m in re.finditer(_pat_eq, texte):
            val = m.group(2).strip()
            if re.match(r'^[a-z]{4,}$', val): # rejeter du texte pur
                continue
            equations.add(f"{m.group(1)}={val}")

        # --------------------------------------------------
        # identification des nombres ou quantités
        # --------------------------------------------------
        # Cherche uniquement des nombres (entiers, décimaux, ou avec séparateur de milliers)
        pattern_nombres = r'\b\d+(?:[.,]\d+)?\b'
        nombres_trouves = re.findall(pattern_nombres, texte)

        nombres_propres = set()
        for nb in nombres_trouves:
            # sécurité : on nettoie les éventuels espaces ou virgules de fin
            nb_propre = nb.strip()
            nombres_propres.add(nb_propre)

        # --------------------------------------------------
        # montants (Budgets, Prix, Économies)
        # --------------------------------------------------
        # Capture les formats : "$32,000", "$1.24 billion", "500 €", "1.5 million $"
        pattern_montants = r'(?:[\$€£]\s*\d+(?:[.,]\d+)*(?:\s*(?:million|billion|trillion))?|\d+(?:[.,]\d+)*(?:\s*(?:million|billion|trillion))?\s*[\$€£])'

        montants_trouves = re.findall(pattern_montants, texte, re.IGNORECASE)
        montants_propres = set(m.strip() for m in montants_trouves)

        # --------------------------------------------------
        # formules mathématiques et autres
        # --------------------------------------------------
        formules_inline = set()

        # statistiques avec espaces : r = -0.12, p < 0.001, F = 166.51
        _pat_stat = (
            r'\b([a-zA-Z]{1,4})'       # variable courte : r, p, F, b, OR, sHR…
            r'\s*([=<>≤≥])\s*'         # opérateur (avec ou sans espaces)
            r'([-−]?\d+(?:[.,]\d+)?)'  # valeur numérique
        )
        for m in re.finditer(_pat_stat, texte):
            var, op, val = m.group(1), m.group(2), m.group(3)
            # ne pas capturer si une unité de mesure suit immédiatement
            apres = texte[m.end():m.end() + 6]
            if re.match(r'\s*(?:mg|kg|ml|Hz|ms|nm|cm|mm|µ|°)', apres, re.IGNORECASE):
                continue
            formules_inline.add(f"{var} {op} {val}")

        # lettres grecques : σ = 0.3, λ = 1.0, β: 0.249, ΔU > 0.225
        _pat_grec = (
            rf'([{self._GRECQUES}][a-zA-Z₀-₉ⱼ]*)'  # lettre grecque + indice optionnel
            r'\s*(?:_[a-zA-Z])?\s*'                  # indice textuel optionnel
            r'([=:<>≤≥])\s*'
            r'([-−]?\d+(?:[.,]\d+)?)'
        )
        for m in re.finditer(_pat_grec, texte):
            formules_inline.add(f"{m.group(1)} {m.group(2)} {m.group(3)}")

        # plus/minus : 60.0 ± 17.6, 50 ± 18
        _pat_pm = r'(\d+(?:[.,]\d+)?)\s*(?:±|\u2009±\u2009)\s*(\d+(?:[.,]\d+)?)'
        for m in re.finditer(_pat_pm, texte):
            formules_inline.add(f"{m.group(1)} ± {m.group(2)}")

        # notation scientifique : 3 × 10⁻⁸, 1.52 × 10⁶, 2.5×10−5
        _pat_sci = (
            r'(\d+(?:[.,]\d+)?)'
            r'\s*×\s*'
            r'10\s*(?:\^|⁻|−|-)?\s*(\d+)'
        )
        for m in re.finditer(_pat_sci, texte):
            formules_inline.add(f"{m.group(1)} × 10^{m.group(2)}")

        # p-values avec inégalité : p < 0.001, p ≤ 0.05
        _pat_pval = r'\bp\s*([<>≤≥])\s*(0\.\d+)'
        for m in re.finditer(_pat_pval, texte):
            formules_inline.add(f"p {m.group(1)} {m.group(2)}")

        return {
            "Noms_Propres":        noms_propres,
            "Mesures":             mesures,
            "References":          references,
            "Molecules_Chimiques": molecules,
            "Equations":           equations,
            "Nombres":             nombres_propres,
            "Montants":            montants_propres,
            "Formules_Inline":     formules_inline,
        }

# ====================================================================================
# classe de comparaison du paragraphe de base et de la réécriture par le LLM
# ====================================================================================
class ComparateurFactuel:

    def comparer(self, faits_source, faits_generes):
        rapport = {
            "alterations_detectees": False,
            "total_erreurs": 0,
            "details": {}
        }

        for categorie in faits_source.keys():
            source_set  = faits_source[categorie]
            genere_set  = faits_generes[categorie]

            suppressions = source_set - genere_set
            ajouts       = genere_set - source_set
            nb_erreurs   = len(suppressions) + len(ajouts)

            rapport["total_erreurs"] += nb_erreurs
            if nb_erreurs > 0:
                rapport["alterations_detectees"] = True

            rapport["details"][categorie] = {
                "suppressions": list(suppressions),
                "ajouts":       list(ajouts)
            }

        return rapport


# ==========================================
# MODE SCAN — TEST VISUEL (optionnel)
# ==========================================
def scanner_dataset(chemin_json):
    """Affiche les extractions article par article pour vérification visuelle."""
    print("Démarrage du scan...")
    extracteur = ExtracteurFactuel()

    with open(chemin_json, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    articles = (raw_data if isinstance(raw_data, list)
                else [a for arts in raw_data.values() for a in arts])
    trouvailles = 0

    for i, article in enumerate(articles):
        for cle in ["Paragraphes", "Paragraphe", "Rew1", "Rew2", "Rew3"]:
            texte_brut = article.get(cle, "")
            if not texte_brut:
                continue

            texte = nettoyer_texte_llm(texte_brut)
            faits = extracteur.extraire(texte)

            if any(faits.values()):
                print(f"\n{'─'*60}")
                print(f"Article {i+1} | Champ : {cle}")
                print(f"{'─'*60}")
                if faits["Noms_Propres"]:
                    print(f"Noms propres        : {faits['Noms_Propres']}")
                if faits["Mesures"]:
                    print(f"Mesures             : {faits['Mesures']}")
                if faits["References"]:
                    print(f"Références          : {faits['References']}")
                if faits["Molecules_Chimiques"]:
                    print(f"Molécules/Cellules  : {faits['Molecules_Chimiques']}")
                if faits["Equations"]:
                    print(f"Équations compactes : {faits['Equations']}")
                if faits["Nombres"]:
                    print(f"Nombres               : {faits['Nombres']}")
                if faits["Montants"]:
                    print(f"Montants               : {faits['Montants']}")
                if faits["Formules_Inline"]:
                    print(f"Formules inline     : {faits['Formules_Inline']}")
                trouvailles += 1

    print(f"\nScan terminé — {trouvailles} passages pertinents détectés.")


# ====================================================================================
# évaluation complète NER + génération du fichier JSON récapitulatif
# ====================================================================================
def evaluer_et_sauvegarder_json(chemin_entree_json, chemin_sortie_json):
    print("Démarrage de l'évaluation globale...")

    extracteur  = ExtracteurFactuel()
    comparateur = ComparateurFactuel()

    with open(chemin_entree_json, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    resultats_finaux   = {}
    articles_a_traiter = []

    if isinstance(raw_data, dict):
        for domaine, articles in raw_data.items():
            articles_a_traiter.extend(articles)
    elif isinstance(raw_data, list):
        articles_a_traiter = raw_data
    else:
        print("Format JSON non reconnu.")
        return

    for article in articles_a_traiter:
        domaine = article.get("Domaine", "Non_Specifie")
        if domaine not in resultats_finaux:
            resultats_finaux[domaine] = []

        texte_source = article.get("Paragraphes", article.get("Paragraphe", ""))
        faits_source = extracteur.extraire(texte_source)

        bloc_article = {
            "DOI":        article.get("Doi", article.get("DOI", "N/A")),
            "Texte_Source": texte_source,
            "Evaluations": {}
        }

        for rew_key in ["Rew1", "Rew2", "Rew3"]:
            texte_brut = article.get(rew_key, "")
            if not texte_brut:
                continue

            texte_propre   = nettoyer_texte_llm(texte_brut)
            faits_generes  = extracteur.extraire(texte_propre)
            rapport        = comparateur.comparer(faits_source, faits_generes)
            rapport["texte_evalue"] = texte_propre
            bloc_article["Evaluations"][rew_key] = rapport

        resultats_finaux[domaine].append(bloc_article)

    with open(chemin_sortie_json, 'w', encoding='utf-8') as f:
        json.dump(resultats_finaux, f, indent=4, ensure_ascii=False)

    print(f"Évaluation terminée ! Résultats sauvegardés : {chemin_sortie_json}")


# ==========================================
# POINT D'ENTRÉE
# ==========================================
if __name__ == "__main__":
    CHEMIN_JSON   = "../data/json/rewrites.json"
    CHEMIN_SORTIE = "../data/json/resultats_complets_ner_new.json"

    # MODE TEST  — affichage visuel pour vérifier les extractions
    scanner_dataset(CHEMIN_JSON)

    # MODE PRODUCTION — décommenter quand les extractions sont validées
    evaluer_et_sauvegarder_json(CHEMIN_JSON, CHEMIN_SORTIE)